# **Imports**

In [ ]:
from pathlib import Path
import logging
import os
import shutil

import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').is_file():
            return p
    raise RuntimeError('Run this notebook from inside the microbiome2function repo.')


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = REPO_ROOT / 'PROT1_MVP'
os.environ.setdefault('HF_HOME', str(WORK_ROOT / 'hf_cache'))

from M2F import (
    configure_logging,
    DatasetInput,
    ProteinGraphInMemoryDataset,
    GraphConvNodeClassifier,
    AAChainEmbedder,
    clean_col,
    encode_go,
    embed_AAsequences,
)

# **Config**

In [ ]:
GO_DEPTH = 2
AA_MODEL_KEY = 'esm2_t6_8M_UR50D'
AA_BATCH_SIZE = 8
FORCE_RELOAD = False

work_root = WORK_ROOT
raw_dir = work_root / 'graph_inmem' / 'raw'
dataset_root = work_root / 'graph_inmem'
checkpoint_dir = work_root / 'checkpoints'

if FORCE_RELOAD:
    shutil.rmtree(dataset_root, ignore_errors=True)

configure_logging(
    logs_dir=str(work_root / 'logs'),
    file_level=logging.DEBUG,
    console_level=logging.INFO,
)
torch.manual_seed(1)

# **Contained Raw Data**

In [ ]:
DEMO_PROTEINS = [
    ('A0A000001', 'MSTNPKPQRKTKRNTNRRPQDVKFPGG', ('GO:0005524', 'GO:0000166')),
    ('A0A000002', 'MADEEKLPPGWEKRMSRSSGRVYYFNHITNASQWERPSGN', ('GO:0003677',)),
    ('A0A000003', 'MKKFFDSRREQIEQIRDKYGKQLS', ('GO:0016787',)),
    ('A0A000004', 'MGDVEKGKKIFIMKCSQCHTVEKGGKHKTGP', ('GO:0003735',)),
    ('A0A000005', 'MSRSLLLRFLLFLLLLPPLP', ('GO:0005524',)),
    ('A0A000006', 'MNNQRKKTARPSFNMLKRARNRVSTV', ('GO:0000166', 'GO:0003677')),
    ('A0A000007', 'MTEITAAMVKELRESTGAGMMDCKNALSETQHE', ('GO:0016787', 'GO:0005524')),
    ('A0A000008', 'MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKSHP', ('GO:0003735',)),
    ('A0A000009', 'MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPT', ('GO:0003677', 'GO:0000166')),
    ('A0A000010', 'MKWVTFISLLFLFSSAYSRGVFRRDTHKSEIAHRFKDLGE', ('GO:0005524',)),
    ('A0A000011', 'MALWMRLLPLLALLALWGPDPAAA', ('GO:0016787',)),
    ('A0A000012', 'MEEPQSDPSVEPPLSQETFSDLWKLLPEN', ('GO:0000166',)),
]


def _go_cell(terms: tuple[str, ...]) -> str:
    return '; '.join(f'[{term}]' for term in terms)


def write_contained_raw_data(raw_dir: Path) -> tuple[Path, Path]:
    raw_dir.mkdir(parents=True, exist_ok=True)
    for stale_chunk in raw_dir.glob('chunk_*.csv'):
        stale_chunk.unlink()

    accession_path = raw_dir / 'uniref_index_count.csv'
    features_path = raw_dir / 'features.csv'

    pd.DataFrame({
        'uniref': [f'UniRef90_{acc}' for acc, _, _ in DEMO_PROTEINS],
        'i': np.arange(1, len(DEMO_PROTEINS) + 1, dtype=np.int64),
    }).to_csv(accession_path, index=False)

    pd.DataFrame({
        'Entry': [acc for acc, _, _ in DEMO_PROTEINS],
        'Sequence': [seq for _, seq, _ in DEMO_PROTEINS],
        'Gene Ontology (molecular function)': [_go_cell(terms) for _, _, terms in DEMO_PROTEINS],
    }).to_csv(features_path, index=False)

    n = len(DEMO_PROTEINS)
    for src in range(1, n + 1):
        pd.DataFrame({
            'j': [(src % n) + 1, ((src + 3 - 1) % n) + 1],
            'v': [1.0, 0.5],
        }).to_csv(raw_dir / f'chunk_{src}.csv', index=False)

    return accession_path, raw_dir


accession_path, edge_dir = write_contained_raw_data(raw_dir)

# **Transforms**

In [ ]:
go_vocab: dict[str, int] = {}
aa_encoder = AAChainEmbedder(
    model_key=AA_MODEL_KEY,
    device='cuda:0' if torch.cuda.is_available() else 'cpu',
)


def _as_multihot(indices, dim: int):
    if not isinstance(indices, tuple) or dim == 0:
        return np.nan
    out = np.zeros(dim, dtype=np.float32)
    if indices:
        out[np.asarray(indices, dtype=np.int64)] = 1.0
    return out if out.sum() > 0 else np.nan


def pre_transform(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()
    df = clean_col(df, 'Sequence', apply_norm=False, apply_strip_pubmed=False, inplace=True)
    df = clean_col(
        df,
        'Gene Ontology (molecular function)',
        apply_norm=False,
        apply_strip_pubmed=True,
        inplace=True,
    )

    df, labels = encode_go(
        df,
        col_name='Gene Ontology (molecular function)',
        depth=GO_DEPTH,
        inplace=True,
    )
    go_vocab.clear()
    go_vocab.update(labels)

    df.loc[:, 'Gene Ontology (molecular function)'] = df[
        'Gene Ontology (molecular function)'
    ].map(lambda indices: _as_multihot(indices, len(go_vocab)))

    return embed_AAsequences(df, embedder=aa_encoder, batch_size=AA_BATCH_SIZE, inplace=True)


def pre_filter(df: pd.DataFrame):
    x_ok = df['Sequence'].map(
        lambda x: isinstance(x, np.ndarray) and x.size > 0 and np.isfinite(x).all()
    )
    y_ok = df['Gene Ontology (molecular function)'].map(
        lambda y: isinstance(y, np.ndarray) and y.size > 0 and np.isfinite(y).all() and y.sum() > 0
    )
    return x_ok & y_ok

# **Dataset**

In [ ]:
inp = DatasetInput(
    path_to_accession_ids_csv_file=accession_path,
    path_to_edge_csv_dir=edge_dir,
    X={'sequence': 'Sequence'},
    Y={'go_f': 'Gene Ontology (molecular function)'},
    request_size=25,
    rps=1.0,
    max_retry=20,
    edge_dst_column='j',
    edge_attr_columns=('v',),
)

ds = ProteinGraphInMemoryDataset(
    root=dataset_root,
    dataset_input=inp,
    pre_transform=pre_transform,
    pre_filter=pre_filter,
    force_reload=FORCE_RELOAD,
    val_set_size=0.15,
    test_set_size=0.15,
)

data = ds[0]
print(data.x.shape, data.edge_index.shape, data.edge_attr.shape, data.y.shape)
assert data.x.size(0) == data.y.size(0) == data.num_nodes
assert data.edge_attr.size(0) == data.edge_index.size(1)

# **Loaders**

In [ ]:
train_loader = ds.train_loader(num_neighbors=[-1, -1], batch_size=4, shuffle=True)
val_loader = ds.val_loader(num_neighbors=[-1, -1], batch_size=4)
test_loader = ds.test_loader(num_neighbors=[-1, -1], batch_size=4)

# **Model**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GraphConvNodeClassifier(
    in_dim=int(data.x.size(-1)),
    edge_dim=int(data.edge_attr.size(-1)),
    msg_dim=128,
    state_dim=128,
    out_dim=int(data.y.size(-1)),
).to(device)

# **Training**

In [ ]:
history = model.fit(
    train=train_loader,
    val=val_loader,
    epochs=5,
    early_stopping=False,
    report_performance_every_kth_epoch=1,
    save_model_to=checkpoint_dir,
)

print(history['best_val_loss'], history['best_model_path'])

# **Test**

In [ ]:
metrics = model.test(test_loader, threshold=0.5)
print(metrics)